# 07 Time Series and Forecasting — Reference Solutions

Complete solutions for the Songbai Nursing Home Legionnaires' disease outbreak time series exercises.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (prevents Chinese labels from showing as tofu boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
cases = df[df["infected"] == 1]

## Question 1: Build a daily hospitalization-count series

In [ ]:
import matplotlib.dates as mdates

# Daily hospitalization counts
hosp_cases = cases[cases["hospitalization_date"].notna()]
hosp_daily = hosp_cases.groupby("hospitalization_date").size()
hosp_daily = hosp_daily.asfreq("D", fill_value=0)
hosp_daily.name = "hospitalizations"

print(f"Series length: {len(hosp_daily)} days")
print(f"Date range: {hosp_daily.index.min().date()} – {hosp_daily.index.max().date()}")
print(f"Total hospitalizations: {hosp_daily.sum()}")

# Add a baseline period
date_range = pd.date_range(
    hosp_daily.index.min() - pd.Timedelta(days=3),
    hosp_daily.index.max() + pd.Timedelta(days=1),
)
hosp_plot = hosp_daily.reindex(date_range, fill_value=0)

# Hospitalization curve + 5-day rolling average
rolling_5 = hosp_daily.rolling(window=5, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    hosp_plot.index, hosp_plot.values,
    width=1.0,
    color="#e34a33", edgecolor="white", linewidth=0.5,
    alpha=0.7, label="Daily hospitalizations",
)
ax.plot(rolling_5.index, rolling_5.values, color="navy", linewidth=2,
        label="5-day rolling average")
ax.set_title(
    "Songbai Nursing Home Legionnaires' Disease Daily Hospitalization Curve + 5-day Rolling Average, January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Hospitalization")
ax.set_ylabel("Number of Hospitalizations")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

## Question 2: Hospitalization forecasting and window comparison

In [ ]:
# Window comparison
print("=== Hospitalization forecast MAE ===")
best_w, best_mae = 3, float("inf")

for w in [3, 5, 7]:
    pred_w = hosp_daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = hosp_daily.loc[pred_w.index]
    mae_w = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w}  MAE={mae_w:.3f}")
    if mae_w < best_mae:
        best_w, best_mae = w, mae_w

print(f"\n→ Best window: window={best_w} (MAE={best_mae:.3f})")

# Actual vs Predicted
pred_best = hosp_daily.rolling(window=best_w).mean().shift(1).dropna()
actual_best = hosp_daily.loc[pred_best.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(actual_best.index, actual_best.values, marker="o", markersize=4,
        label="Actual hospitalizations", color="#e34a33")
ax.plot(pred_best.index, pred_best.values, marker="s", markersize=4,
        label=f"Predicted ({best_w}-day MA)", color="navy", linestyle="--")
ax.set_title(
    f"Songbai Nursing Home Hospitalizations Actual vs Predicted ({best_w}-day Rolling Average), January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date")
ax.set_ylabel("Number of Hospitalizations")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

## Question 3 (challenge): Epidemic curves grouped by severity

In [ ]:
# Build daily onset-count series by severity
severity_levels = ["mild", "moderate", "severe"]
colors = {"mild": "#41b6c4", "moderate": "#fed976", "severe": "#e31a1c"}

# Build daily series (all severities share the date range, including baseline period)
all_onset = cases.groupby("symptom_onset_date").size()
all_onset = all_onset.asfreq("D", fill_value=0)

# Add a baseline period
date_range = pd.date_range(
    all_onset.index.min() - pd.Timedelta(days=3),
    all_onset.index.max() + pd.Timedelta(days=1),
)

severity_daily = {}
for sev in severity_levels:
    sub = cases[cases["clinical_severity"] == sev]
    s = sub.groupby("symptom_onset_date").size()
    severity_daily[sev] = s.reindex(date_range, fill_value=0)

sev_df = pd.DataFrame(severity_daily)

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(sev_df))

for sev in severity_levels:
    ax.bar(
        sev_df.index, sev_df[sev].values, bottom=bottom,
        width=1.0,
        color=colors[sev], edgecolor="white", linewidth=0.5,
        alpha=0.8, label=sev,
    )
    bottom += sev_df[sev].values

ax.set_title(
    "Songbai Nursing Home Legionnaires' Disease Epidemic Curve (Stratified by Severity), January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

# Peak day for each severity
print("=== Peak day by severity ===")
for sev in severity_levels:
    peak_date = sev_df[sev].idxmax()
    peak_count = sev_df[sev].max()
    print(f"  {sev:10s}  peak day = {peak_date.date()}  {peak_count} people that day")

print("\n→ Observe whether severe cases appear in sync with mild ones, or with a time lag")
print("→ If severe cases cluster in the middle of the outbreak, it may mean residents with higher exposure doses fell ill later")

### Interpretation

- **Hospitalization curve**: the hospitalization peak lags the onset peak by a few days; this lag can be used to forecast bed demand
- **Window choice**: a smaller window (3 days) usually performs better in acute clusters because case counts change quickly
- **Severity stratification**: if severe cases concentrate in a particular time window, it may hint at a specific exposure event or a high-risk group
- **Limitation**: the rolling average is the simplest baseline model and cannot capture trend turning points; more advanced methods (such as ARIMA) can improve on this foundation